# Cell 1: run a postgres SQL into docker contrainer.

In [5]:
docker run --name localhost \
  -e POSTGRES_USER=sa \
  -e POSTGRES_PASSWORD=admin123 \
  -e POSTGRES_DB=energy_db \
  -p 5432:5432 \
  -d postgres

SyntaxError: invalid syntax (2251705451.py, line 1)

# Cell 2: Imports and Environment Setup

In [2]:
import os
import sys
import glob
import logging

sys.path.append(os.path.abspath('.'))
from src.schemas.energy_consumption import get_energy_consumption_schema
from src.pipeline.ingestion import PySparkIngestionPipeline

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Cell 3: Pipeline Execution Block

In [3]:
db_url = "jdbc:postgresql://localhost:5432/energy_db"
table_name = "world_energy_consumption"
db_properties = {"user": "sa", "password": "admin123", "driver": "org.postgresql.Driver"}

batch_folder = "Data/pipeline_ingress_batches/"
file_pattern = os.path.join(batch_folder, "world_energy_consumption_batch_1.csv")

schema = get_energy_consumption_schema()

print(schema)

with PySparkIngestionPipeline(schema=schema) as pipeline:
    for file_path in [file_pattern]: #glob.glob(file_pattern):
        # Extract
        raw_df = pipeline.extract(file_path)
        
        # Transform
        transformed_df = pipeline.transform(raw_df)
        
        # Load
        # Uncomment the below line to execute the load operation
        pipeline.load(transformed_df, db_url, table_name, mode="append", properties=db_properties)
        
        logger.info(f"Successfully processed batch: {file_path}")


2026-05-25 15:16:05,757 - INFO - Initializing SparkSession: IngestionPipeline


StructType([StructField('country', StringType(), True), StructField('year', IntegerType(), True), StructField('iso_code', StringType(), True), StructField('population', DoubleType(), True), StructField('gdp', DoubleType(), True), StructField('biofuel_cons_change_pct', DoubleType(), True), StructField('biofuel_cons_change_twh', DoubleType(), True), StructField('biofuel_cons_per_capita', DoubleType(), True), StructField('biofuel_consumption', DoubleType(), True), StructField('biofuel_elec_per_capita', DoubleType(), True), StructField('biofuel_electricity', DoubleType(), True), StructField('biofuel_share_elec', DoubleType(), True), StructField('biofuel_share_energy', DoubleType(), True), StructField('carbon_intensity_elec', DoubleType(), True), StructField('coal_cons_change_pct', DoubleType(), True), StructField('coal_cons_change_twh', DoubleType(), True), StructField('coal_cons_per_capita', DoubleType(), True), StructField('coal_consumption', DoubleType(), True), StructField('coal_elec_p

Ivy Default Cache set to: /Users/jenishzinzuvadiya/.ivy2/cache
The jars for the packages stored in: /Users/jenishzinzuvadiya/.ivy2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d20041f1-8ff6-48ef-ae33-1d3a9a412b1a;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 50ms :: artifacts dl 2ms
	:: modules in use:
	org.checkerframework#checker-qual;3.42.0 from central in [default]
	org.postgresql#postgresql;42.7.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	--------------------